In [ ]:
import numpy as np
from itertools import product

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import StatePreparation, UnitaryGate
from qiskit_aer import AerSimulator


# ============================================================
# CONFIGURATION
# ============================================================

params = ["A", "f", "t", "tau", "phi"]
N_params = len(params)

resolution_list = [4, 4, 4, 4, 4]
velocity_list = [1, 2, 2, 2, 2]

n_samples = 8
shots_covariance = 1E2
n_steps = 3

# Number of integer standard-deviation units retained in the
# discrete Gaussian proposal.
gaussian_cutoff = 1


# ============================================================
# PARAMETER REGISTERS
# ============================================================

n_param_qubits = [
    int(np.ceil(np.log2(r))) for r in resolution_list
]

qreg_param_list = [
    QuantumRegister(n, name=p)
    for p, n in zip(params, n_param_qubits)
]

n_parameter_qubits = sum(n_param_qubits)


# ============================================================
# PARAMETER VALUES
# ============================================================
#
# This is the mapping from a basis-state integer to the actual
# parameter vector.
#
# Replace these arrays with your actual parameter grids.
#
# Each row of parameter_values is one of the 64 basis samples.
#
# ============================================================

rng = np.random.default_rng(1234)

parameter_values = np.zeros(
    (n_samples, N_params),
    dtype=float
)

parameter_values[:, 0] = np.linspace(0.0, 1.0, n_samples)
parameter_values[:, 1] = np.linspace(0.0, 1000.0, n_samples)
parameter_values[:, 2] = np.linspace(0.0, 10.0, n_samples)
parameter_values[:, 3] = np.linspace(0.1, 40.0, n_samples)
parameter_values[:, 4] = np.linspace(0.0, 2*np.pi, n_samples)


# ============================================================
# EXAMPLE TWO DISTRIBUTIONS
# ============================================================
#
# Replace these with your actual amplitudes.
#
# amplitudes_0[i] = amplitude of basis sample i in t=0
# amplitudes_1[i] = amplitude of basis sample i in t=1
#
# ============================================================

mu0 = np.array([0.25, 400.0, 5.0, 8.0, 2.0])
mu1 = np.array([0.35, 500.0, 5.5, 10.0, 2.5])

scale0 = np.array([0.15, 150.0, 1.5, 5.0, 1.0])
scale1 = np.array([0.18, 180.0, 1.7, 6.0, 1.2])

z0 = np.sum(
    ((parameter_values - mu0) / scale0)**2,
    axis=1
)

z1 = np.sum(
    ((parameter_values - mu1) / scale1)**2,
    axis=1
)

probabilities_0 = np.exp(-0.5*z0)
probabilities_1 = np.exp(-0.5*z1)

probabilities_0 /= probabilities_0.sum()
probabilities_1 /= probabilities_1.sum()

amplitudes_0 = np.sqrt(probabilities_0)
amplitudes_1 = np.sqrt(probabilities_1)


# ============================================================
# UTILITIES
# ============================================================

def normalize_amplitudes(amplitudes):
    amplitudes = np.asarray(amplitudes, dtype=complex)
    norm = np.linalg.norm(amplitudes)
    if norm == 0:
        raise ValueError("Amplitude vector has zero norm.")
    return amplitudes / norm


def amplitudes_to_probabilities(amplitudes):
    amplitudes = normalize_amplitudes(amplitudes)
    return np.abs(amplitudes)**2


def exact_mean(amplitudes, values):
    p = amplitudes_to_probabilities(amplitudes)
    return np.sum(p[:, None] * values, axis=0)


# ============================================================
# CROSS-COVARIANCE
# ============================================================
#
# IMPORTANT:
#
# We assume sample i in state 0 is paired with sample i in
# state 1.
#
# Since the two quantum states only contain marginal
# probabilities, we need a classical rule for assigning
# probability to each pair.
#
# "geometric":
#       w_i ~ sqrt(p0_i p1_i)
#
# "uniform":
#       all paired basis samples have equal weight
#
# "minimum":
#       w_i ~ min(p0_i,p1_i)
#
# Geometric is the default because it emphasizes samples that
# have appreciable weight in BOTH distributions.
# ============================================================

def cross_covariance(
    amplitudes_0,
    amplitudes_1,
    parameter_values,
    pairing="geometric"
):
    p0 = amplitudes_to_probabilities(amplitudes_0)
    p1 = amplitudes_to_probabilities(amplitudes_1)

    if pairing == "geometric":
        weights = np.sqrt(p0*p1)
    elif pairing == "minimum":
        weights = np.minimum(p0, p1)
    elif pairing == "uniform":
        weights = np.ones_like(p0)
    else:
        raise ValueError(f"Unknown pairing: {pairing}")

    if np.sum(weights) == 0:
        raise ValueError("No overlap between the two distributions.")

    weights /= np.sum(weights)

    x0 = parameter_values
    x1 = parameter_values

    mean0 = np.sum(weights[:, None] * x0, axis=0)
    mean1 = np.sum(weights[:, None] * x1, axis=0)

    x0_centered = x0 - mean0
    x1_centered = x1 - mean1

    Sigma = np.einsum(
        "i,ip,iq->pq",
        weights,
        x0_centered,
        x1_centered
    )

    return Sigma, mean0, mean1, weights


# ============================================================
# CORRELATION / CS NORMALIZATION
# ============================================================

def covariance_to_correlation(Sigma, epsilon=1e-15):
    var0 = np.maximum(np.abs(np.diag(Sigma)), epsilon)
    var1 = np.maximum(np.abs(np.diag(Sigma)), epsilon)

    denominator = np.sqrt(
        np.outer(var0, var1)
    )

    R = Sigma / denominator
    return np.clip(R, -1.0, 1.0)


# ============================================================
# DISCRETE CORRELATED GAUSSIAN
# ============================================================
#
# We construct
#
#       P(delta) ~ exp(-1/2 delta^T Sigma^-1 delta)
#
# on a finite integer grid.
#
# To avoid unit problems, we first convert Sigma into
# dimensionless grid coordinates.
# ============================================================

def covariance_to_discrete_gaussian(
    Sigma,
    velocity_list,
    cutoff=1
):
    velocity = np.asarray(
        velocity_list,
        dtype=float
    )

    Dinv = np.diag(1.0 / velocity)

    Sigma_grid = Dinv @ Sigma @ Dinv

    # Symmetrize because numerical covariance estimates need not
    # be exactly symmetric.
    Sigma_grid = 0.5 * (
        Sigma_grid + Sigma_grid.T
    )

    # Remove tiny negative eigenvalues caused by finite sampling.
    eigvals, eigvecs = np.linalg.eigh(Sigma_grid)
    eigvals = np.maximum(eigvals, 1e-12)

    Sigma_grid = (
        eigvecs
        @ np.diag(eigvals)
        @ eigvecs.T
    )

    Sigma_inv = np.linalg.pinv(
        Sigma_grid
    )

    values = range(
        -cutoff,
        cutoff + 1
    )

    displacement_vectors = np.array(
        list(product(values, repeat=N_params)),
        dtype=int
    )

    exponent = -0.5 * np.einsum(
        "ni,ij,nj->n",
        displacement_vectors,
        Sigma_inv,
        displacement_vectors
    )

    exponent -= np.max(exponent)

    probabilities = np.exp(exponent)
    probabilities /= probabilities.sum()

    return (
        displacement_vectors,
        probabilities,
        Sigma_grid
    )


# ============================================================
# QUANTUM STATE PREPARATION
# ============================================================

def basis_state_amplitudes_from_probabilities(probabilities):
    return np.sqrt(
        np.asarray(probabilities, dtype=float)
    )


def prepare_parameter_state(
    amplitudes,
    qreg_param_list
):
    qc = QuantumCircuit(*qreg_param_list)

    qc.append(
        StatePreparation(
            normalize_amplitudes(amplitudes)
        ),
        qreg_param_list
    )

    return qc


# ============================================================
# ENCODE BASIS INDEX INTO THE 5 PARAMETER REGISTERS
# ============================================================

def basis_index_to_bits(index, n_qubits_list):
    bits = []
    remaining = int(index)

    for n in reversed(n_qubits_list):
        mask = (1 << n) - 1
        bits.append(
            format(
                remaining & mask,
                f"0{n}b"
            )
        )
        remaining >>= n

    bits.reverse()
    return bits


def basis_index_to_parameter_values(
    index,
    parameter_values
):
    return parameter_values[index]


# ============================================================
# CLASSICAL COVARIANCE ESTIMATION
# ============================================================
#
# This is the hybrid step.
#
# The quantum computer supplies samples from the two
# statevectors. We then compute Sigma classically.
#
# No amplitude estimation or phase estimation is required.
# ============================================================

def estimate_covariance_hybrid(
    amplitudes_0,
    amplitudes_1,
    parameter_values,
    shots=20000,
    pairing="geometric"
):
    simulator = AerSimulator()

    n = int(
        np.log2(len(amplitudes_0))
    )

    q0 = QuantumRegister(n, "data0")
    q1 = QuantumRegister(n, "data1")
    c0 = ClassicalRegister(n, "c0")
    c1 = ClassicalRegister(n, "c1")

    qc = QuantumCircuit(
        q0,
        q1,
        c0,
        c1
    )

    qc.append(
        StatePreparation(
            normalize_amplitudes(amplitudes_0)
        ),
        q0
    )

    qc.append(
        StatePreparation(
            normalize_amplitudes(amplitudes_1)
        ),
        q1
    )

    qc.measure(q0, c0)
    qc.measure(q1, c1)

    compiled = transpile(
        qc,
        simulator,
        optimization_level=1
    )

    result = simulator.run(
        compiled,
        shots=shots
    ).result()

    counts = result.get_counts()

    p0 = np.zeros(
        len(amplitudes_0)
    )

    p1 = np.zeros(
        len(amplitudes_1)
    )

    # The two registers are independently sampled here.
    # We reconstruct their marginal distributions and then
    # impose the specified sample pairing classically.
    for bitstring, count in counts.items():
        pieces = bitstring.replace(" ", "").split()

        if len(pieces) == 2:
            b0, b1 = pieces
        else:
            # Qiskit register ordering can put the two classical
            # registers into one contiguous string.
            full = bitstring.replace(" ", "")
            b0 = full[:n]
            b1 = full[n:]

        i0 = int(b0, 2)
        i1 = int(b1, 2)

        p0[i0] += count
        p1[i1] += count

    p0 /= shots
    p1 /= shots

    Sigma, mean0, mean1, weights = cross_covariance(
        np.sqrt(p0),
        np.sqrt(p1),
        parameter_values,
        pairing=pairing
    )

    return Sigma, mean0, mean1, p0, p1, weights


# ============================================================
# DIRECT STATEVECTOR VERSION
# ============================================================
#
# For simulator development this avoids shot noise.
# On hardware, use estimate_covariance_hybrid() above.
# ============================================================

def estimate_covariance_statevector(
    amplitudes_0,
    amplitudes_1,
    parameter_values,
    pairing="geometric"
):
    Sigma, mean0, mean1, weights = cross_covariance(
        amplitudes_0,
        amplitudes_1,
        parameter_values,
        pairing=pairing
    )

    return Sigma, mean0, mean1, weights


# ============================================================
# PARAMETER SHIFT MATRIX
# ============================================================

def make_shift_matrix(dimension, shift):
    M = np.zeros(
        (dimension, dimension)
    )

    for i in range(dimension):
        j = (i + shift) % dimension
        M[j, i] = 1.0

    return M


# ============================================================
# CORRELATED QUANTUM WALK COIN
# ============================================================

def make_quantum_walk_coin(
    displacement_vectors,
    probabilities
):
    n_moves = len(displacement_vectors)

    n_coin = int(
        np.ceil(np.log2(n_moves))
    )

    coin_state = np.zeros(
        2**n_coin,
        dtype=complex
    )

    coin_state[:n_moves] = np.sqrt(
        probabilities
    )

    coin_state /= np.linalg.norm(
        coin_state
    )

    qcoin = QuantumRegister(
        n_coin,
        "Coin"
    )

    qc = QuantumCircuit(
        qcoin,
        name="CovarianceCoin"
    )

    qc.append(
        StatePreparation(
            coin_state
        ),
        qcoin
    )

    return qc, qcoin


# ============================================================
# CORRELATED SHIFT CIRCUIT
# ============================================================

def make_correlated_shift(
    displacement_vectors,
    qreg_param_list,
    n_param_qubits,
    velocity_list
):
    n_moves = len(
        displacement_vectors
    )

    n_coin = int(
        np.ceil(np.log2(n_moves))
    )

    qcoin = QuantumRegister(
        n_coin,
        "Coin"
    )

    qc = QuantumCircuit(
        *qreg_param_list,
        qcoin,
        name="CorrelatedShift"
    )

    for move_index, displacement in enumerate(
        displacement_vectors
    ):
        binary = format(
            move_index,
            f"0{n_coin}b"
        )

        # Convert |binary> into |11...1>.
        for k, bit in enumerate(binary):
            if bit == "0":
                qc.x(qcoin[k])

        # Apply the SAME displacement vector to all five
        # parameter registers. This is where covariance becomes
        # correlated parameter-space motion.
        for p in range(N_params):
            d = int(displacement[p])

            if d == 0:
                continue

            D = 2**n_param_qubits[p]

            shift = d * int(
                velocity_list[p]
            )

            M = make_shift_matrix(
                D,
                shift
            )

            U = UnitaryGate(
                M,
                label=f"{params[p]}{shift:+d}"
            )

            cU = U.control(
                num_ctrl_qubits=n_coin
            )

            qc.append(
                cU,
                [
                    *qcoin,
                    *qreg_param_list[p]
                ]
            )

        # Undo control-state conversion.
        for k, bit in enumerate(binary):
            if bit == "0":
                qc.x(qcoin[k])

    return qc, qcoin


# ============================================================
# BUILD ONE WALK CIRCUIT
# ============================================================

def build_walk_circuit(
    Sigma,
    initial_parameter_index=0,
    n_steps=1,
    cutoff=1
):
    displacement_vectors, displacement_probabilities, Sigma_grid = (
        covariance_to_discrete_gaussian(
            Sigma,
            velocity_list,
            cutoff=cutoff
        )
    )

    coin_circuit, qcoin = make_quantum_walk_coin(
        displacement_vectors,
        displacement_probabilities
    )

    shift_circuit, _ = make_correlated_shift(
        displacement_vectors,
        qreg_param_list,
        n_param_qubits,
        velocity_list
    )

    qc = QuantumCircuit(
        *qreg_param_list,
        qcoin
    )

    # Initial basis state.
    bits = basis_index_to_bits(
        initial_parameter_index,
        n_param_qubits
    )

    for qreg, bitstring in zip(
        qreg_param_list,
        bits
    ):
        for q, bit in zip(
            qreg,
            bitstring
        ):
            if bit == "1":
                qc.x(q)

    qc.barrier()

    for step in range(n_steps):
        qc.append(
            coin_circuit,
            qcoin
        )

        qc.append(
            shift_circuit,
            [
                *qreg_param_list,
                qcoin
            ]
        )

        if step != n_steps - 1:
            qc.barrier()

    return (
        qc,
        displacement_vectors,
        displacement_probabilities,
        Sigma_grid
    )


# ============================================================
# PRINT COVARIANCE
# ============================================================

def print_covariance(Sigma, title="Covariance"):
    print(f"\n{'='*55}")
    print(title)
    print(f"{'='*55}")

    for i, row in enumerate(Sigma):
        values = " ".join(
            f"{x: .6e}"
            for x in row
        )
        print(f"{params[i]:>4s}: {values}")


# ============================================================
# INITIAL COVARIANCE
# ============================================================

Sigma, mean0, mean1, weights = estimate_covariance_statevector(
    amplitudes_0,
    amplitudes_1,
    parameter_values,
    pairing="geometric"
)

print_covariance(
    Sigma,
    "INITIAL CROSS-COVARIANCE"
)

print("\nMeans:")
for p, m0, m1 in zip(params, mean0, mean1):
    print(f"{p:>4s}: mean_0={m0:.6g}, mean_1={m1:.6g}")


# ============================================================
# WALK LOOP
# ============================================================
#
# This is the hybrid adaptive loop:
#
#   quantum state
#        |
#        v
#   estimate Sigma
#        |
#        v
#   classical covariance -> Gaussian proposal
#        |
#        v
#   quantum correlated walk
#        |
#        v
#   measure final state
#        |
#        v
#   construct new t=0/t=1 states
#        |
#        +----> repeat
#
# In a real implementation, amplitudes_0 and amplitudes_1
# below would be replaced by the states generated by your
# MCMC / parallel-tempering calculation.
# ============================================================

current_amplitudes_0 = amplitudes_0.copy()
current_amplitudes_1 = amplitudes_1.copy()

current_parameter_index = 0

for timestep in range(n_steps):
    print(f"\n{'#'*55}")
    print(f"# TIMESTEP {timestep}")
    print(f"{'#'*55}")

    # --------------------------------------------------------
    # 1. Recompute covariance from CURRENT distributions.
    # --------------------------------------------------------

    Sigma, mean0, mean1, weights = (
        estimate_covariance_statevector(
            current_amplitudes_0,
            current_amplitudes_1,
            parameter_values,
            pairing="geometric"
        )
    )

    print_covariance(
        Sigma,
        f"Sigma at timestep {timestep}"
    )

    # --------------------------------------------------------
    # 2. Build covariance-dependent correlated Gaussian.
    # --------------------------------------------------------

    (
        displacement_vectors,
        displacement_probabilities,
        Sigma_grid
    ) = covariance_to_discrete_gaussian(
        Sigma,
        velocity_list,
        cutoff=gaussian_cutoff
    )

    print(
        f"\nNumber of displacement states: "
        f"{len(displacement_vectors)}"
    )

    print(
        f"Discrete covariance error: "
        f"{np.linalg.norm(Sigma_grid - Sigma, 'fro'):.6e}"
    )

    # --------------------------------------------------------
    # 3. Build quantum walk using CURRENT Sigma.
    # --------------------------------------------------------

    qc_walk, _, _, _ = build_walk_circuit(
        Sigma,
        initial_parameter_index=current_parameter_index,
        n_steps=1,
        cutoff=gaussian_cutoff
    )

    # --------------------------------------------------------
    # 4. Measure only after the walk.
    # --------------------------------------------------------

    c = ClassicalRegister(
        n_parameter_qubits,
        "measurement"
    )

    qc_measured = qc_walk.copy()
    qc_measured.add_register(c)

    qc_measured.measure(
        qc_measured.qubits[
            :n_parameter_qubits
        ],
        c
    )

    simulator = AerSimulator()

    compiled = transpile(
        qc_measured,
        simulator,
        optimization_level=1
    )

    result = simulator.run(
        compiled,
        shots=shots_covariance
    ).result()

    counts = result.get_counts()

    print(
        f"\nNumber of measured states: "
        f"{len(counts)}"
    )

    # --------------------------------------------------------
    # 5. Pick the most probable new parameter state.
    #
    # For a real MCMC implementation you may instead want
    # to retain the entire measured distribution.
    # --------------------------------------------------------

    most_probable_bitstring = max(
        counts,
        key=counts.get
    )

    most_probable_bitstring = (
        most_probable_bitstring.replace(" ", "")
    )

    current_parameter_index = int(
        most_probable_bitstring,
        2
    )

    current_parameter_index %= n_samples

    current_parameter_values = (
        parameter_values[
            current_parameter_index
        ]
    )

    print(
        f"Selected state index: "
        f"{current_parameter_index}"
    )

    print(
        "Selected parameters: "
        + ", ".join(
            f"{p}={v:.6g}"
            for p, v in zip(
                params,
                current_parameter_values
            )
        )
    )

    # --------------------------------------------------------
    # 6. In your actual algorithm, this is where you would
    # construct the NEW t=0 and t=1 quantum states resulting
    # from your parallel-tempering/MCMC evolution.
    #
    # For this self-contained example we leave the states
    # unchanged, so the covariance loop is demonstrable.
    # --------------------------------------------------------

    # current_amplitudes_0 = new_amplitudes_0
    # current_amplitudes_1 = new_amplitudes_1


# ============================================================
# FINAL CIRCUIT
# ============================================================

print(f"\n{'='*55}")
print("FINAL WALK CIRCUIT")
print(f"{'='*55}")

print(f"Parameter qubits: {n_parameter_qubits}")
print(f"Coin qubits: {int(np.ceil(np.log2(len(displacement_vectors))))}")
print(f"Total qubits: {qc_walk.num_qubits}")
print(f"Depth: {qc_walk.depth()}")

# Uncomment if you actually want to draw it.
# qc_walk.draw("mpl")


INITIAL CROSS-COVARIANCE
   A:  5.745232e-03  5.745232e+00  5.745232e-02  2.292348e-01  3.609836e-02
   f:  5.745232e+00  5.745232e+03  5.745232e+01  2.292348e+02  3.609836e+01
   t:  5.745232e-02  5.745232e+01  5.745232e-01  2.292348e+00  3.609836e-01
 tau:  2.292348e-01  2.292348e+02  2.292348e+00  9.146467e+00  1.440324e+00
 phi:  3.609836e-02  3.609836e+01  3.609836e-01  1.440324e+00  2.268127e-01

Means:
   A: mean_0=0.354217, mean_1=0.354217
   f: mean_0=354.217, mean_1=354.217
   t: mean_0=3.54217, mean_1=3.54217
 tau: mean_0=14.2333, mean_1=14.2333
 phi: mean_0=2.22561, mean_1=2.22561

#######################################################
# TIMESTEP 0
#######################################################

Sigma at timestep 0
   A:  5.745232e-03  5.745232e+00  5.745232e-02  2.292348e-01  3.609836e-02
   f:  5.745232e+00  5.745232e+03  5.745232e+01  2.292348e+02  3.609836e+01
   t:  5.745232e-02  5.745232e+01  5.745232e-01  2.292348e+00  3.609836e-01
 tau:  2.292348e-01  2.2